# MLP Implementation

In [122]:
import torch
# 输入(batch_size, input_features)
x = torch.rand(3, 4)
# 参数
W = torch.rand(4, 5)
b = torch.rand(5)
# 输出
y = x @ W + b
print(y.shape)

torch.Size([3, 5])


In [123]:
import torch.nn as nn
# 与torch的线性层比较
linear = nn.Linear(4, 5)
y1 = linear(x)
print(y1.shape)

torch.Size([3, 5])


In [124]:
# linear & mlp from scratch
class Linear(nn.Module):
    def __init__(self, input_features, output_features):
        super().__init__()
        self.W = nn.Parameter(torch.rand(input_features, output_features))
        self.b = nn.Parameter(torch.rand(output_features))
    def forward(self, x):
        return x @ self.W + self.b

class Mlp(nn.Module):
    def __init__(self, input_features, hidden_features,dropout):
        super().__init__()
        self.c_fc = Linear(input_features, hidden_features)
        self.gelu = nn.GELU()
        self.c_proj = Linear(hidden_features, input_features)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x

In [125]:
# test
mlp = Mlp(x.size(1), 8, 0)
y2 = mlp(x)
print(y2.size())

torch.Size([3, 4])


In [126]:
# softmax & crossentropy from scratch
def softmax(x):
    exp_x = torch.exp(x)
    return exp_x / exp_x.sum(dim=1, keepdim=True)

def cross_entropy(logits, target):
    epsilon = 1e-9 
    prob = softmax(logits)
    prob = torch.clamp(prob, epsilon, 1.0-epsilon) # 避免取0或1
    return -torch.log(prob[range(len(target)), target]).mean()

# test, assuming logits = y2
import torch.nn.functional as F
logits = y2
prob = softmax(logits)
print(f'prob:{prob}')
target = torch.tensor([0, 1, 3]) # 指定标签
loss1 = cross_entropy(logits, target)
loss2 = F.cross_entropy(logits, target)
print(f'loss1={loss1}, loss2={loss2}')

prob:tensor([[0.0772, 0.3600, 0.4951, 0.0677],
        [0.0427, 0.4103, 0.5016, 0.0453],
        [0.0365, 0.4190, 0.4485, 0.0960]], grad_fn=<DivBackward0>)
loss1=1.9319967031478882, loss2=1.9319967031478882


In [127]:
# loop
optimizer = torch.optim.SGD(mlp.parameters(), lr=0.01)
for epoch in range(1000):
    optimizer.zero_grad()
    logits = mlp(x)
    loss = cross_entropy(logits, target)
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item():.4f}')

Epoch 0, Loss: 1.9320
Epoch 10, Loss: 1.4862
Epoch 20, Loss: 1.2400
Epoch 30, Loss: 1.1029
Epoch 40, Loss: 1.0229
Epoch 50, Loss: 0.9715
Epoch 60, Loss: 0.9348
Epoch 70, Loss: 0.9060
Epoch 80, Loss: 0.8819
Epoch 90, Loss: 0.8609
Epoch 100, Loss: 0.8419
Epoch 110, Loss: 0.8244
Epoch 120, Loss: 0.8080
Epoch 130, Loss: 0.7925
Epoch 140, Loss: 0.7777
Epoch 150, Loss: 0.7635
Epoch 160, Loss: 0.7499
Epoch 170, Loss: 0.7367
Epoch 180, Loss: 0.7239
Epoch 190, Loss: 0.7114
Epoch 200, Loss: 0.6993
Epoch 210, Loss: 0.6876
Epoch 220, Loss: 0.6761
Epoch 230, Loss: 0.6650
Epoch 240, Loss: 0.6541
Epoch 250, Loss: 0.6435
Epoch 260, Loss: 0.6332
Epoch 270, Loss: 0.6231
Epoch 280, Loss: 0.6133
Epoch 290, Loss: 0.6037
Epoch 300, Loss: 0.5944
Epoch 310, Loss: 0.5853
Epoch 320, Loss: 0.5765
Epoch 330, Loss: 0.5679
Epoch 340, Loss: 0.5595
Epoch 350, Loss: 0.5513
Epoch 360, Loss: 0.5433
Epoch 370, Loss: 0.5355
Epoch 380, Loss: 0.5280
Epoch 390, Loss: 0.5206
Epoch 400, Loss: 0.5134
Epoch 410, Loss: 0.5064
Epo

In [128]:
# inference
with torch.no_grad():
    logits = mlp(x)
    pred = torch.argmax(logits, dim=1)
    print(f'Predicted classes: {pred}')

Predicted classes: tensor([0, 1, 3])
